# GroundLie360 — Index Builder


## §0 — Setup


In [ ]:
import ast
import csv
import json
import re
import collections
from pathlib import Path

csv.field_size_limit(10**7)

REPO_ROOT = Path('..').resolve().parent  # experiments/GroundLie360 -> repo root
DATA_DIR  = REPO_ROOT / 'data'
OUT_DIR   = Path('.')  # same folder as this notebook

DATA_CSV  = DATA_DIR / 'data.csv'
BBOX_CSV  = DATA_DIR / 'bbox.csv'

print('REPO_ROOT:', REPO_ROOT)
print('data.csv exists:', DATA_CSV.exists())
print('bbox.csv exists:', BBOX_CSV.exists())

## §1 — Load data.csv and inspect


In [ ]:
with open(DATA_CSV, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'Total rows: {len(rows)}')
print(f'Columns: {list(rows[0].keys())}')

In [ ]:
# First-level annotation distribution
first_level = collections.Counter(r['first_level_annotation'] for r in rows)
print('first_level_annotation:', dict(first_level))

# Second-level (multilabel) category counts
second_level_counts = collections.Counter()
for r in rows:
    cats = ast.literal_eval(r['second_level_annotation']) if r['second_level_annotation'] else []
    for c in cats:
        second_level_counts[c] += 1
print('\nsecond_level_annotation counts:')
for k, v in second_level_counts.most_common():
    print(f'  {k}: {v}')

# Field coverage
print('\nField coverage:')
for fld in ['false_speech_temporal', 'video_scene_segmentation', 'temporal_edit_location',
             'transcript_concrete', 'title_third_level_annotation', 'video_transcript']:
    non_empty = sum(1 for r in rows if r[fld] and r[fld].strip() not in ['[]', '', 'nan'])
    print(f'  {fld}: {non_empty}/{len(rows)}')

## §2 — Parsing helpers


In [ ]:
def parse_scene_segmentation(s: str) -> list:
    """Parse numpy-style scene segmentation string into list of [start_frame, end_frame] pairs.
    Input:  '[[   1  184]\n [ 185  222]\n ...]'
    Output: [[1, 184], [185, 222], ...]
    """
    if not s or s.strip() in ['[]', '']:
        return []
    pairs = re.findall(r'(\d+)\s+(\d+)', s)
    return [[int(a), int(b)] for a, b in pairs]


def parse_list_field(s: str) -> list:
    """Safe ast.literal_eval with empty fallback."""
    if not s or s.strip() in ['[]', '', 'nan']:
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []


def parse_false_speech_spans(temporal_raw: str) -> list:
    """Returns list of {'start_s': float, 'end_s': float}.
    Input: '[[0, 10.17536]]'
    """
    spans = parse_list_field(temporal_raw)
    return [{'start_s': float(s[0]), 'end_s': float(s[1])} for s in spans if len(s) == 2]


# Sanity checks
assert parse_scene_segmentation('[[   1  184]\n [ 185  222]]') == [[1, 184], [185, 222]]
assert parse_scene_segmentation('[]') == []
assert parse_false_speech_spans('[[0, 10.17536]]') == [{'start_s': 0.0, 'end_s': 10.17536}]
assert parse_list_field('') == []
print('All parsing helpers OK')

In [ ]:
# Verify temporal_edit_mask length = len(scene_frames) + 1 for all rows
mismatches = []
for r in rows:
    scenes = parse_scene_segmentation(r['video_scene_segmentation'])
    mask   = parse_list_field(r['temporal_edit_location'])
    diff   = len(mask) - len(scenes)
    if diff != 1:
        mismatches.append((r['video_id'], len(scenes), len(mask), diff))

if mismatches:
    print(f'WARNING: {len(mismatches)} rows where len(mask) != len(scenes)+1:')
    for m in mismatches[:5]:
        print(' ', m)
else:
    print(f'OK: All {len(rows)} rows have len(mask) == len(scenes)+1')
    print()
    print('Interpretation: temporal_edit_mask[i] = 1 marks the CUT at the START of scene[i]')
    print('  (i.e., the transition from scene[i-1] to scene[i]).')
    print('  For E3: cos(scene_i, scene_{i+1}) is expected to drop when mask[i+1] == 1.')

## §3 — Build `groundlie_index.json`


In [ ]:
# Which video_ids appear in bbox.csv (all CGI videos)
with open(BBOX_CSV, encoding='utf-8') as f:
    bbox_video_ids = set(r['video_id'] for r in csv.DictReader(f))
print(f'video_ids with bboxes: {len(bbox_video_ids)}')

In [ ]:
CAT_MAP = {
    'false_title':          'false_title',
    'false_speech':         'false_speech',
    'temporal_edit':        'temporal_edit',
    'CGI':                  'cgi',
    'contradictory_content':'contradictory',
    'unsupported_content':  'unsupported',
}

index = []
for r in rows:
    raw_id         = r['video_id']
    multilabel_raw = parse_list_field(r['second_level_annotation'])
    multilabel     = [CAT_MAP.get(c, c) for c in multilabel_raw]
    scenes         = parse_scene_segmentation(r['video_scene_segmentation'])
    mask           = parse_list_field(r['temporal_edit_location'])

    entry = {
        # identity
        'raw_id':     raw_id,
        # relative to --data-dir (data/GroundLie360/ on cluster)
        # videos are in vid_groundlie/, CSVs in dataset/
        'video_path': f'vid_groundlie/{raw_id}.mp4',

        # filled from ffprobe on cluster (see §7)
        'duration_seconds': None,
        'fps':              None,

        # labels
        'binary_label':  1 if r['first_level_annotation'] == 'Fake' else 0,
        'multilabel':    multilabel,
        'false_title':   int('false_title'   in multilabel),
        'false_speech':  int('false_speech'  in multilabel),
        'temporal_edit': int('temporal_edit' in multilabel),
        'cgi':           int('cgi'           in multilabel),
        'contradictory': int('contradictory' in multilabel),
        'unsupported':   int('unsupported'   in multilabel),

        # text
        'title': r['title'],

        # E2: false-speech intervals in seconds (already provided by dataset)
        'false_speech_spans': parse_false_speech_spans(r['false_speech_temporal']),

        # E3: scene segmentation in FRAMES (convert to seconds using fps once available)
        # mask[i]=1 means the cut at the start of scene[i] is a temporal edit
        'scene_frames':       scenes,
        'temporal_edit_mask': mask,

        # E4
        'has_bbox': raw_id in bbox_video_ids,

        # convenience
        'has_transcript': bool(
            r['transcript_concrete'] and
            r['transcript_concrete'].strip() not in ['[]', '']
        ),
    }
    index.append(entry)

print(f'Index entries built: {len(index)}')

## §4 — Build `bbox_index.json`


In [ ]:
bbox_index = collections.defaultdict(list)

with open(BBOX_CSV, encoding='utf-8') as f:
    for row in csv.DictReader(f):
        vid_id   = row['video_id']
        frame_id = int(row['frame_id'])
        bbox     = parse_list_field(row['bbox_area'])  # [x, y, w, h]
        if len(bbox) == 4:
            bbox_index[vid_id].append({'frame_id': frame_id, 'bbox': bbox})

# Sort by frame_id within each video
for vid_id in bbox_index:
    bbox_index[vid_id].sort(key=lambda x: x['frame_id'])

print(f'Videos in bbox_index: {len(bbox_index)}')
example_id = next(iter(bbox_index))
print(f'Example ({example_id}): {len(bbox_index[example_id])} frames')
print('First 3 entries:', bbox_index[example_id][:3])

## §5 — Validation and statistics


In [ ]:
import statistics as stats_module

total = len(index)
real  = sum(1 for e in index if e['binary_label'] == 0)
fake  = sum(1 for e in index if e['binary_label'] == 1)

print(f'Total: {total} (Real: {real}, Fake: {fake})')
print()
print('Category breakdown:')
for cat in ['false_title', 'false_speech', 'temporal_edit', 'cgi', 'contradictory', 'unsupported']:
    count = sum(e[cat] for e in index)
    print(f'  {cat}: {count}')

print()
print(f'has_transcript: {sum(e["has_transcript"] for e in index)}/{total}')
print(f'has_bbox:       {sum(e["has_bbox"] for e in index)}/{total}')
print(f'has false_speech_spans: {sum(len(e["false_speech_spans"]) > 0 for e in index)}/{total}')
print(f'has scene_frames:       {sum(len(e["scene_frames"]) > 0 for e in index)}/{total}')

scene_counts = [len(e['scene_frames']) for e in index]
print(f'\nScenes/video: min={min(scene_counts)} mean={stats_module.mean(scene_counts):.1f} max={max(scene_counts)}')
print(f'\nduration_seconds=None: {sum(e["duration_seconds"] is None for e in index)}/{total}')
print('  -> run §7a on cluster then §7b locally to fill these')

In [ ]:
# Spot-checks
cgi_sample = next(e for e in index if e['cgi'] == 1)
print('CGI sample:')
print(f'  raw_id={cgi_sample["raw_id"]}  has_bbox={cgi_sample["has_bbox"]}  in bbox_index={cgi_sample["raw_id"] in bbox_index}')
print(f'  n_bbox_frames={len(bbox_index.get(cgi_sample["raw_id"], []))}')
print()

fs_sample = next(e for e in index if e['false_speech'] == 1)
print('False-speech sample:')
print(f'  raw_id={fs_sample["raw_id"]}  spans={fs_sample["false_speech_spans"]}')
print()

te_sample = next(e for e in index if e['temporal_edit'] == 1)
print('Temporal-edit sample:')
nz = [i for i, v in enumerate(te_sample['temporal_edit_mask']) if v]
print(f'  raw_id={te_sample["raw_id"]}  n_scenes={len(te_sample["scene_frames"])}  nonzero_mask={nz}')
for i in nz:
    if 0 < i <= len(te_sample['scene_frames']):
        prev = te_sample['scene_frames'][i-1]
        curr = te_sample['scene_frames'][i] if i < len(te_sample['scene_frames']) else None
        print(f'  mask[{i}]=1 -> cut between scene[{i-1}]={prev} and scene[{i}]={curr}')

In [ ]:
assert len(index) == len(rows)
ids = [e['raw_id'] for e in index]
assert len(ids) == len(set(ids)), 'Duplicate raw_ids!'
assert sum(e['cgi'] for e in index) == sum(e['has_bbox'] for e in index)
print('All assertions passed.')

## §6 — Save


In [ ]:
OUT_INDEX = OUT_DIR / 'groundlie_index.json'
OUT_BBOX  = OUT_DIR / 'bbox_index.json'

with open(OUT_INDEX, 'w', encoding='utf-8') as f:
    json.dump(index, f, ensure_ascii=False, indent=2)
print(f'Saved: {OUT_INDEX}  ({OUT_INDEX.stat().st_size // 1024} KB)')

with open(OUT_BBOX, 'w', encoding='utf-8') as f:
    json.dump(dict(bbox_index), f, ensure_ascii=False)
print(f'Saved: {OUT_BBOX}  ({OUT_BBOX.stat().st_size // 1024} KB)')

## §7 — Fill `duration_seconds` and `fps` from cluster

### §7a — Submit the SLURM job that extracts video metadata


### §7b — Patch index locally (after git pull)


In [ ]:
VIDEO_META = OUT_DIR / 'video_meta.json'

if not VIDEO_META.exists():
    print('video_meta.json not found — run §7a on the cluster first, then git pull.')
else:
    with open(VIDEO_META) as f:
        video_meta = json.load(f)

    by_id = {e['raw_id']: e for e in index}
    filled = 0
    for raw_id, m in video_meta.items():
        if raw_id in by_id:
            by_id[raw_id]['duration_seconds'] = m.get('duration_seconds')
            by_id[raw_id]['fps']              = m.get('fps')
            if m.get('duration_seconds') is not None:
                filled += 1

    print(f'Patched {filled}/{len(index)} entries with duration/fps')

    over_120 = [e for e in index if e['duration_seconds'] is not None and e['duration_seconds'] > 120]
    keep     = [e for e in index if e['duration_seconds'] is not None and e['duration_seconds'] <= 120]
    no_dur   = [e for e in index if e['duration_seconds'] is None]
    print(f'\nFilter <=120s: keep={len(keep)}  filtered_out={len(over_120)}  unknown={len(no_dur)}')

    if over_120:
        print('\nFiltered-out category breakdown:')
        for cat in ['false_title','false_speech','temporal_edit','cgi','contradictory','unsupported']:
            print(f'  {cat}: {sum(e[cat] for e in over_120)}')

    with open(OUT_INDEX, 'w', encoding='utf-8') as f:
        json.dump(index, f, ensure_ascii=False, indent=2)
    print(f'\nUpdated index saved: {OUT_INDEX}')

### §7c — Optional: save pre-filtered index (<=120s only)


In [ ]:
if (OUT_DIR / 'video_meta.json').exists():
    keep = [e for e in index if e['duration_seconds'] is not None and e['duration_seconds'] <= 120]
    out_filtered = OUT_DIR / 'groundlie_index_filtered.json'
    with open(out_filtered, 'w', encoding='utf-8') as f:
        json.dump(keep, f, ensure_ascii=False, indent=2)
    print(f'Saved filtered index ({len(keep)} entries): {out_filtered}')
else:
    print('Skipped — run §7a and §7b first.')